# Assignment 3 - Convolutional Network and Transfer Learning

The goal of this exercise is to learn how to train a small convolutional neural network (CNN) and fine-tune this trained network in transfer learning tasks.

Our CNN has the following layers:

1. 2D convolutional layer with $Q_1$ channels, kernel size $7\times7$, stride 1 and padding 0.
2. 2D maximum pooling with pooling size $3\times3$ and stride 3
3. `Tanh` activation
4. 2D convolutional layer with $Q_2$ channels, kernel size $5\times5$, stride 1 and padding 2.
5. 2D maximum pooling with pooling size $2\times2$ and stride 2
6. `Tanh` activation
7. A flattening layer to turn the 3D image into a 1D vector
8. A fully-connected layer with the appropriate number of inputs and $O$ outputs.

For this exercise, we will switch to an implementation in `PyTorch`.
We will get used to some concepts in `PyTorch`, such as relying on the `torch.tensor` data structure, implementing the network, the loss functions, the training loop, and accuracy computation, which we will apply to categorical classification.
We will see how backpropagation and weight update will be done automatically by `torch`.

Please make sure that all your variables are compatible with `torch`.
For example, you cannot mix `torch.tensor`s and `numpy.array`s in any part of the code.

The CNN will be trained on the `letters` from EMNIST datasets and fine-tuned with the `digits` from the same dataset.

## Theoretical Sections

In this section, we analyze the key properties of a CNN focusing on its spatial characteristics and parameterization. Specifically, we address the following fundamental aspects:  

1. Receptive Field Computation  
2. Feature Map Dimensions
3. Learnable Parameter Count  
4. Derivatives of Pooling

Through these computations, we gain insights into the network's ability to capture spatial hierarchies and its overall complexity.

Besides, we also point out some possible problems that implementing the training process may face.

#### Task 1.1: Compute Receptive Field

Compute the receptive field size of an element of the final pooling layers before the flattening operation.

Hints:

- Consider one location in the last feature map, i.e., after the second pooling layer. Go backwards through the layers and compute the size of the receptive field for each operation. 
- Look at the graphics from the slides of Lecture 5 to understand how to compute receptive fields for convolution and pooling layers.

Answer:

### Equations 

- Convolution with kernel size $U$: $$F' = (U-1)+F$$
- Pooling with stride and kernel size $S$: $$F' = S\cdot F$$

### Computations

- Initial Feature Map Element: $$F=1$$

- Pooling with Stride and Size $S=2$: $$F'=2\cdot F=2$$

- Convolution with Size $U=5$: $$F''=(5-1+F') = 6$$

- Pooling with Stride and Size $S=3$: $$F'''=3\cdot F'' = 18$$

- Convolution with Size $U=7$: $$F''''=(7-1+F''') = 24$$

- Receptive field in Input: $$24\times24$$


#### Task 1.2: Compute Learnable Parameters

Given that the input image has shape $28\times28$,

1. Compute feature map size, i.e., the number of inputs for the last fully connected layer.
2. Estimate roughly how many learnable parameters this network has by analytically computing and adding the number of parameters in each layer. Express the estimation in terms of $Q_1, Q_2, O$.

Answer:

**1. Compute Feature Map Size (Number of Inputs for Fully Connected Layer)**

We will compute the spatial dimensions of the feature maps after each convolution and pooling layer step-by-step.
Below, $p$ is the padding (which is applied on both sides), $U$ is the size of the kernel and $S$ the stride.


First Convolutional Layer with U=7 and no padding:
  $$
  \left\lfloor\frac{\text{Input} + 2 \cdot p - U}{S}\right\rfloor + 1 = \frac{28 - 7}{1} + 1 = 22 \times 22
  $$

First Max Pooling Layer with stride=3
  $$
  \left\lfloor\frac{\text{Input}-U}{S} \right\rfloor + 1
  = \left\lfloor\frac{22-3}{3}\right\rfloor + 1 = 7 \times 7
  $$

Second Convolutional Layer with U=5 and padding 2:
  $$
  \frac{7 + 2\cdot2 - 5}{1} + 1 = 7 \times 7
  $$

Second Max Pooling Layer with stride 2:
  $$
  \text{Output size} = \left\lfloor\frac{7-2}{2}\right\rfloor +1 = 3 \times 3
  $$

After these operations, the feature map has shape $ 3 \times 3 \times Q_2 $, so that the number of inputs for the fully connected layer is:

$
3 \times 3 \times Q_2 = 9Q_2
$

---

**2. Estimate the Number of Learnable Parameters**

We calculate the number of learnable parameters (weights + biases) for each layer.

First Convolutional Layer
$ (7 \times 7 \times 1) \times Q_1 + Q_1 = 50 Q_1 $

Second Convolutional Layer
$ (5 \times 5 \times Q_1) \times Q_2 + Q_2 = 25 Q_1 Q_2 + Q_2$

Fully Connected Layer
$ (9Q_2+1) \times O$


Final Formula for Total Learnable Parameters
$50 Q_1 + 25 Q_1 Q_2 + Q_2 + 10 Q_2 O$

This formula allows us to estimate the total number of trainable parameters in the network based on the values of $ Q_1 $, $ Q_2 $, and $O$.

When $Q_1=16$, $Q_2=32$ and $O=26$, we have $800+12832+7514=21146$ learnable parameters.

#### Task 1.3: Derivatives of Pooling

Given two pooling methods:

1. Average pooling: $$a_k=\frac1R \sum\limits_{j=1}^R \hat a_{{R(k-1)+j}}$$
2. Maximum pooling: $$a_k=\max\limits_{j \in \{1,\ldots, R\}} \hat a_{{R(k-1)+j}}$$

Compute $$\frac{\partial a_k}{\partial \hat a_{R(k-1)+j}}$$

Answer:

Average pooling:$$\frac{\partial a_k}{\partial \hat a_{R(k-1)+j}} = \frac {1}{R}$$
Maximum pooling:

$$\frac{\partial a_k}{\partial \hat a_{R(k-1)+j}} = \begin{cases} 1& \text{if } \hat a_{R(k-1)+j} = a_k\\ 0 & \text{otherwise}\end{cases}$$


#### Task 1.4

For a randomly initialized network, what is the estimated probabilities for the classes in binary classification and categorical classification tasks?
To which expected loss value would this turn to?

Make use of the loss functions provided below:

$$
\mathcal{J}^{\text{BCE}} = - \frac{1}{N} \sum_{n=1}^{N} \left[ t^{[n]} \log(y^{[n]}) + (1 - t^{[n]}) \log(1 - y^{[n]}) \right]
$$

$$
\mathcal{J}^{\text{CCE}} = - \frac{1}{N} \sum_{n=1}^{N} \sum_{o=1}^{O} t_o^{[n]} \log y_o^{[n]}
$$


Answer:

**Binary cross-entropy loss**

For an uninitialized network, the output before training is usually around 0.5 due to symmetric initialization (e.g., Xavier or Kaiming).

Assuming random outputs where $ y^{[n]} \approx 0.5 $ for all samples, the expected loss can be computed as:

$$
\mathbb{E}[\mathcal{J}^{\text{BCE}}] = - \left[ t \log 0.5 + (1 - t) \log 0.5 \right]= -t \log 0.5 - \log 0.5 + t \log 0.5= - \log 0.5 \approx 0.693
$$


---

**Categorical cross-entropy loss**

For an uninitialized network, the output logits are typically similar, resulting in:

$$
y_o^{[n]} \approx \frac{1}{O}
$$

Substituting this into the loss formula:

$$
\mathbb{E}[\mathcal{J}^{\text{CCE}}] = - \sum_{o=1}^{O} t_o \log \frac{1}{O} = - \log \frac{1}{O} = \log O
$$



#### Task 1.5:

Given the example loss and accuracy plots, compared with the standard plots, in which the learning rate is 0.001 and the computation for loss and accuracy is correct, analyze the possible problems for the other plots.

Answer:

1. Validation loss and accuracy oscillate: Learning rate is too high and overfitting is possible.

2. Loss decreases slowly and accuracy increases slowly: Learning rate is too low and causes a slow convergence and requires more iterations

3. Loss value is too small but accuracy is normal: dividing the sum of average batch loss by the number of training samples
